# Elliptic Bitcoin graph components

This notebook loads the Elliptic Bitcoin dataset into `data/EllipticBitcoinDataset`, finds weakly connected components, and visualizes sampled connected subgraphs colored by label: licit, illicit, and unknown.

In [3]:
%load_ext autoreload
%autoreload 2

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / "graph_structure.py").exists() else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from script.graph_structure import (
    build_undirected_graph,
    build_directed_graph,
    component_summary,
    load_elliptic_data,
    plot_connected_components,
)

DATA_ROOT = PROJECT_ROOT / "data" / "EllipticBitcoinDataset"
OUTPUT_DIR = PROJECT_ROOT / "figures" / "elliptic_components"
DATA_ROOT, OUTPUT_DIR

c:\Users\20TBa\.conda\envs\btctran\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(WindowsPath('d:/Github/BitcoinTransactionClassification/data/EllipticBitcoinDataset'),
 WindowsPath('d:/Github/BitcoinTransactionClassification/figures/elliptic_components'))

In [4]:
data = load_elliptic_data(DATA_ROOT)
graph = build_undirected_graph(data)

print(data)
print(f"Connected components: {graph.number_of_nodes():,} nodes, {graph.number_of_edges():,} undirected edges")

summary_df = pd.DataFrame(component_summary(graph, data, limit=10))
summary_df

Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], train_mask=[203769], test_mask=[203769])
Connected components: 203,769 nodes, 234,355 undirected edges


,rank,nodes,edges,licit,illicit,unknown
0,1,7880,9164,2130,17,5733
1,2,7140,8493,1915,239,4986
2,3,6803,8623,1874,8,4921
3,4,6727,8588,954,18,5755
4,5,6621,8316,1268,11,5342
5,6,6393,7813,1675,33,4685
6,7,6048,7253,1101,102,4845
7,8,5894,7014,1605,158,4131
8,9,5693,8180,1410,30,4253
9,10,5598,6673,1216,5,4377


In [5]:
paths = plot_connected_components(
    data_root=DATA_ROOT,
    output_dir=OUTPUT_DIR,
    num_components=4,
    max_nodes_per_component=300,
)
paths

[WindowsPath('d:/Github/BitcoinTransactionClassification/figures/elliptic_components/component_01.png'),
 WindowsPath('d:/Github/BitcoinTransactionClassification/figures/elliptic_components/component_02.png'),
 WindowsPath('d:/Github/BitcoinTransactionClassification/figures/elliptic_components/component_03.png'),
 WindowsPath('d:/Github/BitcoinTransactionClassification/figures/elliptic_components/component_04.png')]

In [ ]:
from IPython.display import Image, display

for path in paths:
    display(Image(filename=str(path)))

We now examine some properties on the edge connections.

In [25]:
from script.graph_structure import LABEL_NAMES

labels = data.y.cpu().numpy()
graph_labeled = build_undirected_graph(data, only_labeled=True)

rows = []

for u in graph_labeled.nodes:
    one_hop_neighbors = set(graph_labeled.neighbors(u))
    illicit_one_hop = [nbr for nbr in one_hop_neighbors if labels[nbr] == 1]

    # note this implementation does not count connections via unlabeled nodes
    within_two_hops = set()
    for nbr in one_hop_neighbors:
        within_two_hops.update(set(graph_labeled.neighbors(nbr)))

    exact_two_hops = within_two_hops - one_hop_neighbors - {u}
    illicit_two_hop = [nbr for nbr in exact_two_hops if labels[nbr] == 1]

    rows.append(
        {
            "node": int(u),
            "node_label": int(labels[u]),
            "node_label_name": LABEL_NAMES[int(labels[u])],
            "labeled_neighbor_count": len(one_hop_neighbors),
            "one_hop_illicit_count": len(illicit_one_hop),
            "one_hop_illicit_share": (
                len(illicit_one_hop) / len(one_hop_neighbors) if one_hop_neighbors else np.nan
            ),
            "two_hop_labeled_count": len(exact_two_hops),
            "two_hop_illicit_count": len(illicit_two_hop),
            "two_hop_illicit_share": (
                len(illicit_two_hop) / len(exact_two_hops) if exact_two_hops else np.nan
            ),
            "has_illicit_1hop": int(len(illicit_one_hop) > 0),
            "has_illicit_2hop": int(len(illicit_two_hop) > 0),
        }
    )

neighborhood_eda = pd.DataFrame(rows)
neighborhood_eda.head(5)

,node,node_label,node_label_name,labeled_neighbor_count,one_hop_illicit_count,one_hop_illicit_share,two_hop_labeled_count,two_hop_illicit_count,two_hop_illicit_share,has_illicit_1hop,has_illicit_2hop
0,3,0,licit,62,1,0.016129,72,0,0.0,1,0
1,9,0,licit,23,0,0.000000,24,0,0.0,0,0
2,10,0,licit,2,0,0.000000,1,0,0.0,0,0
3,11,0,licit,2,0,0.000000,3,0,0.0,0,0
4,16,0,licit,2,0,0.000000,245,0,0.0,0,0


In [28]:
summary_by_node_label = (
    neighborhood_eda.groupby("node_label_name")
    .agg(
        nodes=("node", "size"),
        mean_labeled_neighbor_count=("labeled_neighbor_count", "mean"),
        mean_one_hop_illicit_count=("one_hop_illicit_count", "mean"),
        mean_one_hop_illicit_share=("one_hop_illicit_share", "mean"),
        share_with_any_illicit_1hop=("has_illicit_1hop", "mean"),
        mean_two_hop_illicit_count=("two_hop_illicit_count", "mean"),
        mean_two_hop_illicit_share=("two_hop_illicit_share", "mean"),
        share_with_any_illicit_2hop=("has_illicit_2hop", "mean"),
    )
    .round(4)
)

summary_by_node_label

,nodes,mean_labeled_neighbor_count,mean_one_hop_illicit_count,mean_one_hop_illicit_share,share_with_any_illicit_1hop,mean_two_hop_illicit_count,mean_two_hop_illicit_share,share_with_any_illicit_2hop
node_label_name,,,,,,,,
illicit,4545,0.8123,0.4392,0.5097,0.3001,8.8018,0.6799,0.6964
licit,42019,1.6553,0.0404,0.0181,0.0244,0.2177,0.0184,0.0712


In [30]:
labeled_src, labeled_dst = map(list, zip(*graph_labeled.edges))

src_labels = [LABEL_NAMES[int(labels[u])] for u in labeled_src]
dst_labels = [LABEL_NAMES[int(labels[v])] for v in labeled_dst]

transition_counts = pd.crosstab(
    pd.Series(src_labels, name="source_label"),
    pd.Series(dst_labels, name="target_label"),
).reindex(index=["licit", "illicit"], columns=["licit", "illicit"], fill_value=0)

transition_counts

target_label,licit,illicit
source_label,,
licit,33930,1228
illicit,468,998


In [ ]:
# Edge homophily: fraction of labeled directed edges whose endpoint labels match
edge_homophily = np.mean([labels[u] == labels[v] for u, v in graph_labeled.edges])
edge_homophily.item()

0.9536915683704674